# Selection des features

***Résumé Exécutif du Feature Engineering - Prévision Solaire J+1***

L'étape de feature_engineering a consisté à la prise en compte des conclusions de l'analyse exploratoire (EDA), ainsi que la création de garde-fous données manquantes/outliers ainsi que la création de features cycliques et de variables laggées.

**Données** : Dataset horaire sur 2.5 ans (01/2023 - 05/2025) de production RTE et prévisions OpenMétéo. Identification de 2 valeurs manquantes nocturnes dans la cible, qui seront imputées à 0.

**Gardes-fous :**
- Création de tests IQR/Z-score et raise des outliers en intersection des deux filtres ;
- Interpolation des séquences temporelles inférieures à 3h consécutives ;
- Lors d'une absence de séquence de plus de 3 heures, création d'un reporting des séquences les plus longues, et potentiellement création d'un futur algorithme KNN - Filtre de Kalman.

**Features créées :**
- Création de features cycliques heures + mois, en fonction de la saisonnalité du cycle solaire ;
- Création de features laggées (data leakage évité): 
  - Retard de 1, 6, 12, 24 et 48 (observation des cross-correlation + cohérent physiquement) ;
  - Moyennes mobiles de 6, 12, 24 et 48 périodes (analogue aux features retard) ;
  - Ramp de 6, 12, 24 et 48 périodes.
 
**Etapes effectuées dans ce notebook** :

- Sélection de features agnostique (Embedding via LightGBM) ;

### Import des librairies

In [3]:
# Registry
from dotenv import dotenv_values

# ML / DL
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.base import TransformerMixin, BaseEstimator

# Visualisation
import seaborn as sns
from tqdm import tqdm 

#Logs, typing, libraries standards
import logging 
import joblib
import warnings
from typing import Dict, List, Any, Tuple, Set, Optional
import pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")

# Config basique logging
PROJECT_ROOT = Path().resolve().parent
logging.basicConfig(filename=PROJECT_ROOT / "logs/data_training.log", level=logging.INFO, datefmt="%Y-%m-%d-%H-%M-%s")
logging.getLogger('mlflow').setLevel(logging.ERROR)

#### Préparation des datasets d'entrainement

La stratégie est la suivante : 
- SARIMAX dispose de la série brute en MWh en entrée, car ce n'est qu'une baseline ;
- La série brute est ensuite normalisée par le décile 99 glissant sur les 90 derniers jours. Il suit l'augmentation de la capacité du parc, et n'est que data-driven, nous assurant que la tendance est bien incluse dedans.

In [4]:
# Data core
df = pd.read_csv(PROJECT_ROOT / "data/processed/training_dataset.csv", index_col=0)
df.index = pd.to_datetime(df.index, utc=True).tz_convert("Europe/Paris")
meteo_features = joblib.load(PROJECT_ROOT / "data/processed/features/meteo_features.pkl")

# Features
X = df.drop(columns="solar_mw")
y = df["solar_mw"]

# Séparation train test split
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.15,
                                                    shuffle=False, 
                                                    random_state=42)

# Configuratio secrets
config = dotenv_values(PROJECT_ROOT / ".env")

#### Pareto feature selection
Nous allons utiliser une embedded method par LightGBM pour sélectionner les features par gain et importance (seuil 95%). Cela va nous permettre de discriminer, avec la connaissance physiques, les variables à utiliser dans le modèle LGBM, et globalement donner une idée des contributions de chaque variable.

In [6]:
def pareto_feature_selection(X: pd.DataFrame, 
                            y: pd.Series, 
                            params: Dict[str, Any], 
                            horizon: int, 
                            gain_threshold: float) -> pd.DataFrame: 
    """"""
    
    # Initialisation des décalages des features
    y_shifted = y.shift(-horizon).dropna()
    X_aligned = X.loc[y_shifted.index]
    
    # Selector
    model = lgb.LGBMRegressor(**params)
    selector = model.fit(X=X_aligned, y=y_shifted)
    
    # Construction de la matrice de gain
    gain_matrix = pd.DataFrame(data={"Features": X.columns, "Gain": selector.feature_importances_})
    gain_matrix = gain_matrix.sort_values(by="Gain", ascending=False)
    gain_matrix["cumulative_gain"] = (gain_matrix["Gain"] / gain_matrix["Gain"].sum()).cumsum()
    h_features = gain_matrix.loc[gain_matrix["cumulative_gain"] <= gain_threshold]

    return h_features

In [7]:
def horizon_feature_selection(X: pd.DataFrame, 
                              y: pd.Series, 
                              params: Dict[str, Any], 
                              horizons: List[int],
                              threshold: float) -> Dict[str, List[str]]:
    """Return the feature selected by Pareto threshold"""
    selected_features = {}
    
    for horizon in horizons:
        selected_features[f'features_t+{horizon}'] = pareto_feature_selection(X=X, 
                                                y=y, 
                                                params=params, 
                                                horizon=horizon, 
                                                gain_threshold=threshold)
        
    return selected_features

In [8]:
def ensure_cyclic_features(feature_sets: dict[str, list[str]], 
                            cyclic_features: list[str],
                            X: pd.DataFrame) -> dict[str, list[str]]:
    """
    Garantit la présence de toutes les features cycliques dans chaque set,
    à condition qu'elles existent bien dans X.
    """
    available_cyclic = [f for f in cyclic_features if f in X.columns]
    
    return {
        set_name: list(set(features) | set(available_cyclic))
        for set_name, features in feature_sets.items()
    }

In [36]:
def union_features(*feature_lists: List[str]) -> List[str]:
    return list(set().union(*feature_lists))

In [ ]:
# Feature selection
params = {
    'num_leaves': 61,              
    'max_depth': -1,               
    'learning_rate': 0.01,         
    'n_estimators': 500,
    'min_child_samples': 20,
    'verbosity': -1,
    'random_state': 42,
    'importance_type': 'gain',     
    'n_jobs': -1
}
threshold = 0.95
horizons = [1, 3, 12, 24]

# Running
selected_features = horizon_feature_selection(
    X=X_train, 
    y=y_train, 
    params=params, 
    horizons=horizons, 
    threshold=threshold
    )

In [ ]:
feature_sets = {
    "short": union_features(selected_features["features_t+1"]["Features"],  selected_features["features_t+3"]["Features"]), # type: ignore
    "mid":   union_features(selected_features["features_t+3"]["Features"],  selected_features["features_t+12"]["Features"]), # type: ignore
    "long":  union_features(selected_features["features_t+12"]["Features"], selected_features["features_t+24"]["Features"]), # type: ignore
}

# Features cycliques complémentaires
CYCLIC_FEATURES = ["hour_sin", "hour_cos", "month_sin", "month_cos"]
feature_sets = ensure_cyclic_features(feature_sets, CYCLIC_FEATURES, X_train)
joblib.dump(feature_sets, PROJECT_ROOT / "data/processed/features/lgbm_selected_features.pkl")

['data/processed/features/selected_features.plk']

Nous pouvons remarquer les choses suivantes :
- l'horizon t+1 est dominé par les features d'autorégression : ramp, lag, variables cycliques et global irradiance ;
- l'horizon t+3 est un intermédiaire : les lag t+6 sont intéressants, avec les features physiques qui commencent à prendre de l'ampleur ;
- l'horizon t+12 et t+24 montrent que la situation passée le jour d'avant ainsi que les prévisions météos deviennent importantes.

Proposition : 
- Prendre comme noyau les variables l'union des deux bornes sans doublons à tous les horizons ;
- Possiblement ajouter des features physiques lorsque cela est pertinent, ou des variables qui fonctionnent en double (cos, sin, temperature...)

L'objectif est de proposer un comparatif des modèles LGBM/LSTM avec baseline naïve et SARIMAX h+1: 
- Court terme h+1 à h+3 ;
- Moyen-terme h+4 à h+12 ;
- Long terme H+13 à h+24.

## Modèle LSTM seq2seq

Il est proposé comme benchmark DL d'utiliser un seq2seq afin de prendre en compte de manière intéressante la structure historique/prévision de la production solaire.

### Model agnostic feature selection

On utilise le Pareto LGBM comme filtre grossier, et réduire le nombre de features à un nombre gérable. Le seuil sera plus permissif (0.96), car le modèle LGBM peut supprimer des features potentiellement utile à la compréhension de la série temporelle par le LSTM. Le but étant d'avoir maximum 50 - 80 variables, pour ne pas overfitter le modèle.

In [ ]:
# Feature selection
params = {
    'num_leaves': 61,              
    'max_depth': -1,               
    'learning_rate': 0.01,         
    'n_estimators': 500,
    'min_child_samples': 20,
    'verbosity': -1,
    'random_state': 42,
    'importance_type': 'gain',     
    'n_jobs': -1
}
threshold = 0.96
horizons = [1, 6, 12, 24]

# Running
lstm_selected_features = horizon_feature_selection(
    X=X_train, 
    y=y_train, 
    params=params, 
    horizons=horizons, 
    threshold=threshold
    )

In [ ]:
feature_sets = [set(features["Features"].values) for features in lstm_selected_features.values()]
final_lstm_features = set().union(*feature_sets)
joblib.dump(final_lstm_features, PROJECT_ROOT / "data/processed/features/lstm_selected_features.pkl")

['data/processed/features/lstm_selected_features.plk']

In [39]:
len(final_lstm_features)

60

60 variables semble pas trop important pour limiter l'overfitting.
Les valeurs prévisionnelles sont annotées d'un "forecast". il suffit donc de filtrer pour trouver les variables décoders. Les variables encoders sont donc le résultat de la soustraction des variables décoders.

In [ ]:
lstm_features = joblib.load(PROJECT_ROOT / "data/processed/features/lstm_selected_features.pkl")
len(lstm_features)

60

In [ ]:
def filter_seq2seq_features(lstm_features: Set[str]) -> Tuple[List[str], List[str]]:
    """Return encoder features and decoder features"""
    
    #Init
    decoder_features = []
    encoder_features = []
    
    for feature in lstm_features:
        
        if "forecast" in feature:
            decoder_features.append(feature)

        elif "month" in feature or "hour" in feature:
            decoder_features.append(feature)
            encoder_features.append(feature)

        else:
            encoder_features.append(feature)

    return (encoder_features, decoder_features)

In [42]:
encoder_features, decoder_features = filter_seq2seq_features(lstm_features=lstm_features)

In [ ]:
joblib.dump(encoder_features, PROJECT_ROOT / "data/processed/features/encoder_features.pkl")
joblib.dump(decoder_features, PROJECT_ROOT / "data/processed/features/decoder_features.pkl")

['data/processed/features/decoder_features.plk']

Il est possible de remarquer que le decoder a beaucoup plus de features que l'encoder, ce qui est logique car les prévisions sont plus importantes que le passé.

### Orchestrateur

L'orchestrateur wrappe les fonctions précédemment proposées en héritant des classes abstraites Sklearn.

In [ ]:
class LGBMFeatureSelector(BaseEstimator, TransformerMixin):
    """
    Agnostic feature selector based on gain importance of LGBM.
    Select variables according to Pareto front on several temporal horizon.
    """

    def __init__(
            self,
            lgbm_params: Dict[str, Any],
            horizons: List[int],
            threshold: float = 0.95,
            always_keep: Optional[List[str]] = None
        ):
        
        # initialization
        self.lgbm_params = lgbm_params
        self.horizons = horizons
        self.threshold = threshold
        self.always_keep = always_keep or []

        # Dynamic learned parameters
        self.selected_features_per_horizon_: Dict[str, List[str]] = {}
        self.final_selected_features_: List[str] = []
        self._is_fitted: bool = False

    def fit(self, X: pd.DataFrame, y: pd.Series):
        """Train a LGBM model for each horizon to memorize best features set respectively"""

        all_selected_set = set()

        for horizon in tqdm(self.horizons):
            
            # Temporal alignment
            y_shifted = y.shift(-horizon).dropna()
            X_aligned = X.loc[y_shifted.index]

            # Training
            model = lgb.LGBMRegressor(**self.lgbm_params)
            model.fit(X=X_aligned, y=y_shifted)

            # Gain
            gain_df = ( 
                pd.DataFrame({'Feature': X.columns, 'Gain': model.feature_importances_})
                .sort_values(by="Gain", ascending=False)
            )
            gain_df["cumulative_gain"] = (gain_df["Gain"] / gain_df["Gain"].sum()).cumsum()
            
            # Pareto selector
            selected_cols = (
                gain_df.loc[gain_df["cumulative_gain"] < self.threshold, "Feature"]
                .to_list()
            )

            # Metadata stockage
            self.selected_features_per_horizon_[f"h_{horizon}"] = selected_cols
            all_selected_set.update(selected_cols)

        # Needed features
        features_kept = [f for f in self.always_keep if f in X.columns]
        all_selected_set.update(features_kept)

        # Stateful transformer
        self.final_selected_features_ = list(all_selected_set)
        self._is_fitted = True

        return self     

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """Select useful variables on a new DataFrame"""
        if not self._is_fitted:
            raise ValueError("Selector has not bee fitted. Call .fit() before transform values")
        
        return X[self.final_selected_features_]

In [6]:
params = {
    'num_leaves': 61,              
    'max_depth': -1,               
    'learning_rate': 0.01,         
    'n_estimators': 500,
    'min_child_samples': 20,
    'verbosity': -1,
    'random_state': 42,
    'importance_type': 'gain',     
    'n_jobs': -1
}
threshold = 0.95
horizons = [1, 3, 12, 24]
always_keep = ["month_sin", "month_cos", "hour_sin", "hour_cos", "day_sin", "day_cos"]

In [7]:
instance = LGBMFeatureSelector(
    lgbm_params=params, 
    horizons=horizons, 
    always_keep = always_keep
    )

In [8]:
instance.fit(X_train, y_train)

100%|██████████| 4/4 [01:12<00:00, 18.01s/it]


,lgbm_params,"{'importance_type': 'gain', 'learning_rate': 0.01, 'max_depth': -1, 'min_child_samples': 20, ...}"
,horizons,"[1, 3, ...]"
,threshold,0.95
,always_keep,"['month_sin', 'month_cos', ...]"


In [9]:
X_useful = instance.transform(X=X_train)

In [10]:
X_useful.head()

,day_sin,hour_cos,global_tilted_irradiance_forecast_t+16,solar_mw_volatility_6,solar_mw_lag_t-1,global_tilted_irradiance_forecast_t+6,precipitation_std_forecast_delta_t+24_t,global_tilted_irradiance_forecast_delta_t+12_t,global_tilted_irradiance_forecast_delta_t+1_t,global_tilted_irradiance_std_forecast_t+3,...,global_tilted_irradiance_delta_minmax_forecast_t+12,global_tilted_irradiance_delta_minmax_forecast_t+1,day_cos,hour_sin,global_tilted_irradiance_std_forecast_t+12,month_cos,temperature_2m_forecast_delta_t+6_t,global_tilted_irradiance_delta_minmax_forecast_t+24,cloud_cover_forecast_t+24,global_tilted_irradiance_std_forecast_t+16
2023-02-03 02:00:00+01:00,0.05162,0.86603,49.999996,0.0,0.0,0.00000,0.0,494.999970,0.0,0.0000,...,95.2469,0.0,0.99867,0.50000,30.7552,0.5,-0.75,0.0,0.0,12.2468
2023-02-03 03:00:00+01:00,0.05162,0.70711,0.000000,0.0,0.0,35.00000,0.0,439.999970,0.0,0.0000,...,94.2284,0.0,0.99867,0.70711,29.7176,0.5,-0.65,0.0,0.0,0.0000
2023-02-03 04:00:00+01:00,0.05162,0.50000,0.000000,0.0,0.0,165.99997,0.0,335.000000,0.0,0.0000,...,82.9630,0.0,0.99867,0.86603,26.0156,0.5,0.45,0.0,0.0,0.0000
2023-02-03 05:00:00+01:00,0.05162,0.25882,0.000000,0.0,0.0,320.99997,0.0,194.000020,0.0,0.0000,...,64.0000,0.0,0.99867,0.96593,20.6807,0.5,1.65,0.0,0.0,0.0000
2023-02-03 06:00:00+01:00,0.05162,0.00000,0.000000,0.0,0.0,429.00000,0.0,49.999996,0.0,7.9576,...,42.0000,0.0,0.99867,1.00000,12.2468,0.5,2.20,0.0,0.0,0.0000
